# Sold Week 7

In [2]:
import pandas as pd

sold = pd.read_csv('../../IDX_Data/sold_week6_feature_engineered.csv')
print('Initial shape:', sold.shape)

/var/folders/hx/s9px6scd2pl5dqfp0pldbdgc0000gn/T/ipykernel_16887/2131255401.py:3: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: ListAgentEmail, 3: FireplaceYN, 4: BuyerAgencyCompensationType, 5: latfilled, 6: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv('../../IDX_Data/sold_week6_feature_engineered.csv')


Initial shape: (394847, 76)


In [3]:
def get_iqr_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return lower, upper

In [4]:
# ClosePrice
cp_low, cp_high = get_iqr_bounds(sold['ClosePrice'])
sold['closeprice_outlier_flag'] = (
    (sold['ClosePrice'] < cp_low) | 
    (sold['ClosePrice'] > cp_high)
)

# LivingArea
la_low, la_high = get_iqr_bounds(sold['LivingArea'])
sold['livingarea_outlier_flag'] = (
    (sold['LivingArea'] < la_low) | 
    (sold['LivingArea'] > la_high)
)

# DaysOnMarket
dom_low, dom_high = get_iqr_bounds(sold['DaysOnMarket'])
sold['daysonmarket_outlier_flag'] = (
    (sold['DaysOnMarket'] < dom_low) | 
    (sold['DaysOnMarket'] > dom_high)
)

In [5]:
print('ClosePrice outliers:', sold['closeprice_outlier_flag'].sum())
print('LivingArea outliers:', sold['livingarea_outlier_flag'].sum())
print('DaysOnMarket outliers:', sold['daysonmarket_outlier_flag'].sum())

ClosePrice outliers: 28761
LivingArea outliers: 17427
DaysOnMarket outliers: 28982


In [8]:
sold_filtered = sold[
    (~sold['closeprice_outlier_flag']) &
    (~sold['livingarea_outlier_flag']) &
    (~sold['daysonmarket_outlier_flag'])
]

print('Rows BEFORE:', sold.shape[0])
print('Rows AFTER:', sold_filtered.shape[0])

Rows BEFORE: 394847
Rows AFTER: 334493


In [9]:
print('--- Median Comparison ---')

print('ClosePrice BEFORE:', sold['ClosePrice'].median())
print('ClosePrice AFTER:', sold_filtered['ClosePrice'].median())

print('LivingArea BEFORE:', sold['LivingArea'].median())
print('LivingArea AFTER:', sold_filtered['LivingArea'].median())

print('DaysOnMarket BEFORE:', sold['DaysOnMarket'].median())
print('DaysOnMarket AFTER:', sold_filtered['DaysOnMarket'].median())

--- Median Comparison ---
ClosePrice BEFORE: 820000.0
ClosePrice AFTER: 785000.0
LivingArea BEFORE: 1641.0
LivingArea AFTER: 1568.0
DaysOnMarket BEFORE: 19.0
DaysOnMarket AFTER: 16.0


In [10]:
sold.to_csv('../../IDX_Data/sold_week7_flagged.csv', index=False)
sold_filtered.to_csv('../../IDX_Data/sold_week7_filtered.csv', index=False)

print('Week 7 sold datasets saved.')

Week 7 sold datasets saved.


# Summary
- Applied IQR method to detect outliers
- Created flag columns for ClosePrice, LivingArea, DaysOnMarket
- Preserved full dataset with outlier flags
- Created filtered dataset removing extreme values
- Compared dataset size and median values before and after filtering